# Deepfake Detector - full Colab pipeline
**Runtime -> Change runtime type -> T4 GPU**, then **Runtime -> Run all**.

Steps: install -> write project files -> offline smoke test -> download dataset -> prepare -> train -> inspect results -> launch web demo.

In [ ]:
!pip install -q timm grad-cam kagglehub gradio scikit-learn
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch runtime to T4 GPU")

## 1. Write the project files

In [ ]:
import os
os.makedirs("/content/deepfake_detector/dfd", exist_ok=True)
os.makedirs("/content/deepfake_detector/app", exist_ok=True)
os.makedirs("/content/deepfake_detector/tests", exist_ok=True)
os.makedirs("/content/deepfake_detector/models", exist_ok=True)
os.chdir("/content/deepfake_detector")
print(os.getcwd())

In [ ]:
%%writefile dfd/__init__.py
# package

In [ ]:
%%writefile dfd/preprocessing.py
"""Face detection & preprocessing (OpenCV Haar cascade, no extra weights needed)."""
from __future__ import annotations

import cv2
import numpy as np

IMG_SIZE = 224
_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)


def resize_square(img_rgb: np.ndarray, size: int = IMG_SIZE) -> np.ndarray:
    """Resize without face detection (for datasets that are already face crops)."""
    if img_rgb.shape[:2] == (size, size):
        return img_rgb
    return cv2.resize(img_rgb, (size, size), interpolation=cv2.INTER_AREA)


def to_model_input(face_rgb_uint8: np.ndarray):
    """(H,W,3) RGB uint8 -> (3,H,W) ImageNet-normalised float tensor."""
    import torch
    img = face_rgb_uint8.astype(np.float32) / 255.0
    img = (img - _MEAN) / _STD
    return torch.from_numpy(img).permute(2, 0, 1).contiguous().float()


class FacePreprocessor:
    def __init__(self, margin_ratio: float = 0.3):
        path = cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
        self.detector = cv2.CascadeClassifier(path)
        if self.detector.empty():
            raise RuntimeError(f"Could not load face cascade from {path}")
        self.margin_ratio = margin_ratio

    def _find_faces(self, gray):
        gray = cv2.equalizeHist(gray)
        # strict pass first, then a relaxed pass for small / low-contrast faces
        for scale, neigh, min_size in [(1.1, 5, (60, 60)), (1.05, 3, (30, 30))]:
            faces = self.detector.detectMultiScale(gray, scaleFactor=scale,
                                                   minNeighbors=neigh, minSize=min_size)
            if len(faces) > 0:
                return faces
        return []

    def detect_and_crop(self, image_bgr: np.ndarray, fallback_full: bool = False):
        """Returns a square (IMG_SIZE,IMG_SIZE,3) RGB uint8 crop of the largest face.
        If no face is found: None, or the whole image resized if fallback_full=True."""
        if image_bgr is None or image_bgr.size == 0:
            return None
        H, W = image_bgr.shape[:2]
        faces = self._find_faces(cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY))
        rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
        if len(faces) == 0:
            return self._square_resize(rgb) if fallback_full else None

        x, y, w, h = max(faces, key=lambda f: f[2] * f[3])
        side = int(max(w, h) * (1 + 2 * self.margin_ratio))
        side = max(1, min(side, H, W))
        cx, cy = x + w // 2, y + h // 2
        x0 = int(np.clip(cx - side // 2, 0, W - side))
        y0 = int(np.clip(cy - side // 2, 0, H - side))
        crop = rgb[y0:y0 + side, x0:x0 + side]
        return cv2.resize(crop, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)

    @staticmethod
    def _square_resize(rgb):
        H, W = rgb.shape[:2]
        s = min(H, W)
        y0, x0 = (H - s) // 2, (W - s) // 2
        return cv2.resize(rgb[y0:y0 + s, x0:x0 + s], (IMG_SIZE, IMG_SIZE),
                          interpolation=cv2.INTER_AREA)

    to_model_input = staticmethod(to_model_input)


def extract_frames(video_path: str, every_n: int = 15, max_frames: int = 64):
    """Every Nth frame of a video (BGR), capped at max_frames."""
    cap = cv2.VideoCapture(video_path)
    frames, idx = [], 0
    while len(frames) < max_frames:
        ok, frame = cap.read()
        if not ok:
            break
        if idx % every_n == 0:
            frames.append(frame)
        idx += 1
    cap.release()
    return frames

In [ ]:
%%writefile dfd/model.py
"""Dual-branch detector: spatial (EfficientNet) + frequency (FFT -> small CNN)."""
from __future__ import annotations

import timm
import torch
import torch.nn as nn


class FrequencyBranch(nn.Module):
    def __init__(self, out_dim: int = 128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.BatchNorm2d(16), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d(1),
        )
        self.proj = nn.Linear(64, out_dim)

    @staticmethod
    def to_frequency_input(x: torch.Tensor) -> torch.Tensor:
        """(B,3,H,W) -> (B,1,H,W) per-sample-standardised log-magnitude spectrum.
        Always float32: cuFFT half precision only supports power-of-2 sizes (224 isn't)."""
        with torch.autocast(device_type=x.device.type, enabled=False):
            gray = x.float().mean(dim=1, keepdim=True)
            mag = torch.abs(torch.fft.fftshift(torch.fft.fft2(gray), dim=(-2, -1)))
            log_mag = torch.log1p(mag)
            flat = log_mag.flatten(1)
            flat = (flat - flat.mean(1, keepdim=True)) / (flat.std(1, keepdim=True) + 1e-6)
            return flat.view_as(log_mag)

    def forward(self, x):
        f = self.to_frequency_input(x)
        return self.proj(self.net(f).flatten(1))


class SpatialBranch(nn.Module):
    def __init__(self, backbone_name="efficientnet_b0", pretrained=True, out_dim=256):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, pretrained=pretrained, num_classes=0)
        self.proj = nn.Linear(self.backbone.num_features, out_dim)

    def forward(self, x):
        return self.proj(self.backbone(x))


class DeepfakeDetector(nn.Module):
    """Returns one logit per image; sigmoid(logit) = P(fake)."""

    def __init__(self, backbone_name="efficientnet_b0", pretrained=True):
        super().__init__()
        self.backbone_name = backbone_name
        self.spatial = SpatialBranch(backbone_name, pretrained, 256)
        self.frequency = FrequencyBranch(128)
        self.classifier = nn.Sequential(
            nn.Linear(256 + 128, 128), nn.ReLU(inplace=True), nn.Dropout(0.3), nn.Linear(128, 1)
        )

    def forward(self, x):
        return self.classifier(torch.cat([self.spatial(x), self.frequency(x)], dim=1))

    @torch.no_grad()
    def predict_proba(self, x):
        return torch.sigmoid(self.forward(x)).squeeze(-1)


def save_checkpoint(model: DeepfakeDetector, path: str, **meta):
    torch.save({"state_dict": model.state_dict(), "backbone": model.backbone_name, **meta}, path)


def load_checkpoint(path: str, device="cpu") -> DeepfakeDetector:
    """Loads checkpoints made by save_checkpoint (also accepts a bare state_dict)."""
    ckpt = torch.load(path, map_location=device)
    if isinstance(ckpt, dict) and "state_dict" in ckpt:
        state, backbone = ckpt["state_dict"], ckpt.get("backbone", "efficientnet_b0")
    else:
        state, backbone = ckpt, "efficientnet_b0"
    model = DeepfakeDetector(backbone_name=backbone, pretrained=False)
    model.load_state_dict(state)
    return model.to(device).eval()

In [ ]:
%%writefile dfd/dataset.py
"""Dataset: <root>/real/*.(jpg|png)  (label 0)  and  <root>/fake/*  (label 1)."""
from __future__ import annotations

import random
from pathlib import Path

import cv2
import numpy as np
import torch
from torch.utils.data import Dataset

from .preprocessing import IMG_SIZE, resize_square, to_model_input

VALID_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


class FaceCropDataset(Dataset):
    def __init__(self, root_dir: str, augment: bool = False):
        self.root = Path(root_dir)
        self.augment = augment
        self.samples: list[tuple[Path, int]] = []
        for label, folder in [(0, "real"), (1, "fake")]:
            d = self.root / folder
            if d.exists():
                self.samples += [(f, label) for f in sorted(d.iterdir())
                                 if f.suffix.lower() in VALID_EXTS]
        if not self.samples:
            raise RuntimeError(f"No images found in {self.root}/real or {self.root}/fake")

    def __len__(self):
        return len(self.samples)

    @property
    def n_real(self):
        return sum(1 for _, l in self.samples if l == 0)

    @property
    def n_fake(self):
        return len(self.samples) - self.n_real

    @staticmethod
    def _augment(img):
        """Simulate re-upload 'laundering' (compression/blur/resize) + flip."""
        if random.random() < 0.5:
            q = random.randint(40, 90)
            ok, enc = cv2.imencode(".jpg", cv2.cvtColor(img, cv2.COLOR_RGB2BGR),
                                   [cv2.IMWRITE_JPEG_QUALITY, q])
            if ok:
                img = cv2.cvtColor(cv2.imdecode(enc, cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)
        if random.random() < 0.3:
            k = random.choice([3, 5])
            img = cv2.GaussianBlur(img, (k, k), 0)
        if random.random() < 0.3:  # down/up-scale
            s = random.randint(112, 200)
            img = cv2.resize(cv2.resize(img, (s, s), interpolation=cv2.INTER_AREA),
                             (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR)
        if random.random() < 0.5:
            img = np.ascontiguousarray(img[:, ::-1])
        return img

    def _load(self, idx):
        """Read an image, skipping unreadable files instead of crashing training."""
        for off in range(len(self.samples)):
            path, label = self.samples[(idx + off) % len(self.samples)]
            bgr = cv2.imread(str(path))
            if bgr is not None:
                return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB), label
        raise RuntimeError("All images are unreadable")

    def __getitem__(self, idx):
        img, label = self._load(idx)
        img = resize_square(img)
        if self.augment:
            img = self._augment(img)
        return to_model_input(img), torch.tensor(label, dtype=torch.float32)

In [ ]:
%%writefile dfd/prepare_data.py
"""Build data/{train,val}/{real,fake} from a downloaded dataset.

Two modes:
  * pre-cropped datasets (e.g. 140k Real and Fake Faces): symlink files, no re-encoding
  * raw photos / video frames: --detect finds + crops faces (saved as lossless PNG)
"""
from __future__ import annotations

import argparse
import os
import random
import shutil
from pathlib import Path

import cv2

from .dataset import VALID_EXTS
from .preprocessing import FacePreprocessor

SPLIT_ALIASES = {"train": ["train", "training"], "val": ["valid", "val", "validation"],
                 "test": ["test", "testing"]}


def _images(d: Path):
    return sorted(f for f in d.iterdir() if f.suffix.lower() in VALID_EXTS)


def find_splits(base: str) -> dict:
    """Walk `base` and find dirs like .../<train|valid|test>/{real,fake}. Layout-agnostic."""
    found = {}
    for root, dirs, _ in os.walk(base):
        names = {d.lower(): d for d in dirs}
        if "real" in names and "fake" in names:
            split_name = Path(root).name.lower()
            for key, aliases in SPLIT_ALIASES.items():
                if split_name in aliases and key not in found:
                    found[key] = (Path(root) / names["real"], Path(root) / names["fake"])
    return found


def build_split(real_dir, fake_dir, out_dir, n_per_class, detect=False, seed=0):
    rng = random.Random(seed)
    out = Path(out_dir)
    pre = FacePreprocessor() if detect else None
    counts = {}
    for label, src in [("real", Path(real_dir)), ("fake", Path(fake_dir))]:
        dst = out / label
        if dst.exists():
            shutil.rmtree(dst)
        dst.mkdir(parents=True)
        files = _images(src)
        rng.shuffle(files)
        n_ok = 0
        for f in files:
            if n_ok >= n_per_class:
                break
            if detect:
                img = cv2.imread(str(f))
                crop = pre.detect_and_crop(img) if img is not None else None
                if crop is None:
                    continue
                cv2.imwrite(str(dst / f"{n_ok:05d}.png"), cv2.cvtColor(crop, cv2.COLOR_RGB2BGR))
            else:
                target = dst / f"{n_ok:05d}{f.suffix.lower()}"
                try:
                    os.symlink(f.resolve(), target)
                except OSError:
                    shutil.copy2(f, target)
            n_ok += 1
        counts[label] = n_ok
    print(f"{out}: {counts}")
    return counts


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--src", required=True, help="downloaded dataset root")
    p.add_argument("--out", default="data")
    p.add_argument("--n_train", type=int, default=2000, help="images per class")
    p.add_argument("--n_val", type=int, default=500)
    p.add_argument("--detect", action="store_true", help="run face detector + crop")
    a = p.parse_args()

    splits = find_splits(a.src)
    print("Found splits:", {k: (str(v[0]), str(v[1])) for k, v in splits.items()})
    if "train" not in splits:
        raise SystemExit("Could not find a train/{real,fake} folder under --src")
    val = splits.get("val") or splits.get("test")
    if val is None:
        raise SystemExit("Could not find a valid/ or test/ folder under --src")
    build_split(*splits["train"], f"{a.out}/train", a.n_train, a.detect)
    build_split(*val, f"{a.out}/val", a.n_val, a.detect, seed=1)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile dfd/train.py
"""Training:  python -m dfd.train --train_dir data/train --val_dir data/val --epochs 10"""
from __future__ import annotations

import argparse
import json
import random
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import (accuracy_score, f1_score, precision_score,
                             recall_score, roc_auc_score)
from torch.utils.data import DataLoader

from .dataset import FaceCropDataset
from .model import DeepfakeDetector, load_checkpoint, save_checkpoint


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


@torch.no_grad()
def evaluate(model, loader, device) -> dict:
    model.eval()
    probs, labels = [], []
    for x, y in loader:
        p = torch.sigmoid(model(x.to(device, non_blocking=True))).squeeze(-1).float().cpu()
        probs += p.tolist()
        labels += y.tolist()
    preds = [int(p >= 0.5) for p in probs]
    m = {
        "accuracy": accuracy_score(labels, preds),
        "precision": precision_score(labels, preds, zero_division=0),
        "recall": recall_score(labels, preds, zero_division=0),
        "f1": f1_score(labels, preds, zero_division=0),
        "auc": roc_auc_score(labels, probs) if len(set(labels)) > 1 else float("nan"),
    }
    return m


def train(args):
    set_seed(args.seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    use_amp = device.type == "cuda"
    print(f"Device: {device} | AMP: {use_amp}")

    train_ds = FaceCropDataset(args.train_dir, augment=True)
    val_ds = FaceCropDataset(args.val_dir, augment=False)
    kw = dict(num_workers=args.workers, pin_memory=use_amp,
              persistent_workers=args.workers > 0)
    train_loader = DataLoader(train_ds, batch_size=args.batch_size, shuffle=True,
                              drop_last=len(train_ds) > args.batch_size, **kw)
    val_loader = DataLoader(val_ds, batch_size=args.batch_size, shuffle=False, **kw)
    print(f"Train: {len(train_ds)} (real {train_ds.n_real}/fake {train_ds.n_fake}) | "
          f"Val: {len(val_ds)}")

    model = DeepfakeDetector(args.backbone, pretrained=not args.no_pretrained).to(device)
    pos_w = torch.tensor([train_ds.n_real / max(train_ds.n_fake, 1)], device=device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_w)  # handles class imbalance
    opt = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=args.epochs)
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    Path(args.out).parent.mkdir(parents=True, exist_ok=True)
    best, bad_epochs, history = -1.0, 0, []
    for epoch in range(1, args.epochs + 1):
        model.train()
        t0, total, n = time.time(), 0.0, 0
        for x, y in train_loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True).unsqueeze(1)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, enabled=use_amp):
                loss = criterion(model(x).float(), y)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
            total += loss.item() * x.size(0)
            n += x.size(0)
        sched.step()

        vm = evaluate(model, val_loader, device)
        history.append({"epoch": epoch, "train_loss": total / max(n, 1), **vm})
        print(f"Epoch {epoch}/{args.epochs} | loss {total / max(n, 1):.4f} | "
              f"acc {vm['accuracy']:.3f} | auc {vm['auc']:.3f} | f1 {vm['f1']:.3f} | "
              f"{time.time() - t0:.0f}s")

        score = vm["auc"] if not np.isnan(vm["auc"]) else vm["accuracy"]
        if score > best:
            best, bad_epochs = score, 0
            save_checkpoint(model, args.out, epoch=epoch, val_metrics=vm)
            print(f"  -> saved best ({best:.3f}) to {args.out}")
        else:
            bad_epochs += 1
            if args.patience and bad_epochs >= args.patience:
                print(f"Early stopping (no improvement for {args.patience} epochs).")
                break

    results = {"best_val_score": best, "history": history}
    if args.ood_dir:
        model = load_checkpoint(args.out, device)
        ood = evaluate(model, DataLoader(FaceCropDataset(args.ood_dir), batch_size=args.batch_size,
                                         num_workers=args.workers), device)
        results["ood"] = ood
        results["auc_gap"] = best - ood["auc"]
        print("\n--- Generalisation check (unseen generator) ---")
        print({k: round(v, 3) for k, v in ood.items()}, "| AUC gap:", round(results["auc_gap"], 3))

    with open(Path(args.out).with_suffix(".json"), "w") as f:
        json.dump(results, f, indent=2)
    print(f"\nDone. Best val score: {best:.3f}")
    return results


def get_args(argv=None):
    p = argparse.ArgumentParser()
    p.add_argument("--train_dir", default="data/train")
    p.add_argument("--val_dir", default="data/val")
    p.add_argument("--ood_dir", default=None)
    p.add_argument("--backbone", default="efficientnet_b0")
    p.add_argument("--epochs", type=int, default=10)
    p.add_argument("--batch_size", type=int, default=32)
    p.add_argument("--lr", type=float, default=3e-4)
    p.add_argument("--workers", type=int, default=2)
    p.add_argument("--patience", type=int, default=4, help="0 disables early stopping")
    p.add_argument("--seed", type=int, default=42)
    p.add_argument("--no_pretrained", action="store_true", help="skip ImageNet weights (offline tests)")
    p.add_argument("--out", default="models/detector_best.pt")
    return p.parse_args(argv)


if __name__ == "__main__":
    train(get_args())

In [ ]:
%%writefile dfd/explain.py
"""Grad-CAM explainability on the spatial branch's last conv layer."""
from __future__ import annotations

import cv2
import numpy as np
import torch
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

from .model import DeepfakeDetector

# (y0, y1, x0, x1) as fractions of the 224x224 crop; overlap is fine, we take the max mean.
FACE_REGIONS = {
    "eyes / brow area": (0.15, 0.42, 0.10, 0.90),
    "nose / cheek area": (0.40, 0.65, 0.20, 0.80),
    "mouth / jaw area": (0.60, 0.90, 0.15, 0.85),
    "hairline / forehead edge": (0.0, 0.18, 0.0, 1.0),
}


def interpret(heatmap: np.ndarray, prob: float):
    h, w = heatmap.shape
    scores = {}
    for name, (y0, y1, x0, x1) in FACE_REGIONS.items():
        r = heatmap[int(y0 * h):int(y1 * h), int(x0 * w):int(x1 * w)]
        scores[name] = float(r.mean()) if r.size else 0.0
    top = max(scores, key=scores.get)
    if prob >= 0.7:
        note = f"Likely manipulated - the strongest signal is around the {top}."
    elif prob >= 0.5:
        note = f"Possibly manipulated - some signal around the {top}, low confidence."
    else:
        note = "No strong manipulation signal - consistent with an authentic image."
    return top, note


def overlay_heatmap(face_rgb_uint8: np.ndarray, heatmap: np.ndarray, alpha: float = 0.45):
    color = cv2.applyColorMap(np.uint8(255 * np.clip(heatmap, 0, 1)), cv2.COLORMAP_JET)
    color = cv2.cvtColor(color, cv2.COLOR_BGR2RGB)
    return (alpha * color + (1 - alpha) * face_rgb_uint8).astype(np.uint8)


class DeepfakeExplainer:
    def __init__(self, model: DeepfakeDetector, device: str = "cpu"):
        self.device = device
        self.model = model.to(device).eval()
        backbone = model.spatial.backbone
        layer = getattr(backbone, "conv_head", None)
        if layer is None:  # other timm backbones: last child that outputs a feature map
            layer = [m for m in backbone.children() if isinstance(m, torch.nn.Module)][-2]
        self.cam = GradCAM(model=self.model, target_layers=[layer])

    def explain(self, face_tensor: torch.Tensor) -> dict:
        """face_tensor: (3,224,224) normalised. Returns prob, heatmap, top_region, note."""
        batch = face_tensor.unsqueeze(0).to(self.device)
        with torch.no_grad():
            prob = torch.sigmoid(self.model(batch)).item()
        heat = self.cam(input_tensor=batch, targets=[ClassifierOutputTarget(0)])[0]
        top, note = interpret(heat, prob)
        return {"probability_fake": prob, "heatmap": heat, "top_region": top, "note": note}

    # kept for backwards-compatible call style
    overlay_heatmap = staticmethod(overlay_heatmap)

In [ ]:
%%writefile dfd/pipeline.py
"""One entry point used by every front-end (Streamlit, Gradio, notebook, CLI)."""
from __future__ import annotations

from pathlib import Path

import numpy as np
import torch

from .explain import DeepfakeExplainer, overlay_heatmap
from .model import DeepfakeDetector, load_checkpoint
from .preprocessing import FacePreprocessor, extract_frames, to_model_input

DEFAULT_WEIGHTS = Path(__file__).resolve().parent.parent / "models" / "detector_best.pt"


class DeepfakePipeline:
    def __init__(self, weights: str | Path | None = None, device: str | None = None):
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        weights = Path(weights) if weights else DEFAULT_WEIGHTS
        self.weights_loaded = weights.exists()
        if self.weights_loaded:
            self.model = load_checkpoint(str(weights), self.device)
        else:  # untrained model so the UI can still be exercised
            self.model = DeepfakeDetector(pretrained=False).to(self.device).eval()
        self.pre = FacePreprocessor()
        self.explainer = DeepfakeExplainer(self.model, self.device)

    def analyze_image(self, image_rgb: np.ndarray) -> dict:
        """image_rgb: (H,W,3) uint8 RGB. Returns dict; 'face_found' False if no face."""
        bgr = np.ascontiguousarray(image_rgb[:, :, ::-1])
        face = self.pre.detect_and_crop(bgr)
        if face is None:
            return {"face_found": False}
        res = self.explainer.explain(to_model_input(face))
        res.update(face_found=True, face=face,
                   overlay=overlay_heatmap(face, res["heatmap"]),
                   weights_loaded=self.weights_loaded)
        return res

    @torch.no_grad()
    def analyze_video(self, path: str, every_n: int = 15, max_frames: int = 32) -> dict:
        faces = []
        for fr in extract_frames(path, every_n, max_frames):
            f = self.pre.detect_and_crop(fr)
            if f is not None:
                faces.append(to_model_input(f))
        if not faces:
            return {"face_found": False}
        probs = []
        for i in range(0, len(faces), 16):
            batch = torch.stack(faces[i:i + 16]).to(self.device)
            probs += torch.sigmoid(self.model(batch)).squeeze(-1).cpu().tolist()
        return {"face_found": True, "frames_used": len(probs), "mean_prob": float(np.mean(probs)),
                "max_prob": float(np.max(probs)), "per_frame": probs,
                "weights_loaded": self.weights_loaded}

In [ ]:
%%writefile app/gradio_app.py
"""Gradio UI (works in Colab with a public link):  python app/gradio_app.py"""
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))

import gradio as gr

from dfd.pipeline import DeepfakePipeline


def build_demo(weights=None):
    pipe = DeepfakePipeline(weights)

    def run(image):
        if image is None:
            return None, None, None, "Upload an image."
        res = pipe.analyze_image(image)
        if not res["face_found"]:
            return None, None, None, "No face detected - try a clearer, front-facing photo."
        p = res["probability_fake"]
        warn = "" if pipe.weights_loaded else "\n(WARNING: untrained model - score is meaningless)"
        return res["face"], res["overlay"], {"Fake": p, "Real": 1 - p}, res["note"] + warn

    return gr.Interface(
        fn=run, inputs=gr.Image(type="numpy", label="Photo"),
        outputs=[gr.Image(label="Detected face"), gr.Image(label="Grad-CAM heatmap"),
                 gr.Label(label="Verdict"), gr.Textbox(label="Explanation")],
        title="Deepfake Detector",
        description="Upload a face photo. Treat the result as a signal, not a verdict.",
    )


if __name__ == "__main__":
    build_demo().launch(share=True)

In [ ]:
%%writefile tests/smoke_test.py
"""End-to-end offline sanity check (synthetic data, no downloads):  python tests/smoke_test.py"""
import shutil
import sys
import tempfile
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))

import cv2
import numpy as np
import torch

from dfd.dataset import FaceCropDataset
from dfd.explain import DeepfakeExplainer
from dfd.model import DeepfakeDetector, load_checkpoint
from dfd.pipeline import DeepfakePipeline
from dfd.preprocessing import FacePreprocessor, to_model_input
from dfd.train import get_args, train


def make_split(root, n):
    rng = np.random.default_rng(0)
    for label in ["real", "fake"]:
        (root / label).mkdir(parents=True, exist_ok=True)
        for i in range(n):
            base = rng.integers(0, 255, (224, 224, 3), dtype=np.uint8)
            if label == "fake":  # crude periodic pattern so the task is learnable
                base[::4] = 255
            cv2.imwrite(str(root / label / f"{i}.jpg"), base)


def main():
    tmp = Path(tempfile.mkdtemp())
    try:
        make_split(tmp / "train", 8)
        make_split(tmp / "val", 4)

        ds = FaceCropDataset(str(tmp / "train"), augment=True)
        x, y = ds[0]
        assert x.shape == (3, 224, 224), x.shape
        print("dataset OK", len(ds))

        m = DeepfakeDetector(pretrained=False).eval()
        assert m(torch.randn(2, 3, 224, 224)).shape == (2, 1)
        print("model OK")

        out = str(tmp / "ckpt.pt")
        train(get_args(["--train_dir", str(tmp / "train"), "--val_dir", str(tmp / "val"),
                        "--epochs", "1", "--batch_size", "4", "--workers", "0",
                        "--no_pretrained", "--out", out]))
        load_checkpoint(out)
        print("train + checkpoint OK")

        face = np.random.randint(0, 255, (224, 224, 3), dtype=np.uint8)
        res = DeepfakeExplainer(m).explain(to_model_input(face))
        assert res["heatmap"].shape == (224, 224)
        print("grad-cam OK:", res["note"])

        pipe = DeepfakePipeline(weights=out)
        r = pipe.analyze_image(face)  # noise has no face -> must not crash
        assert r["face_found"] in (True, False)
        print("pipeline OK")
        print("\nALL SMOKE TESTS PASSED")
    finally:
        shutil.rmtree(tmp, ignore_errors=True)


if __name__ == "__main__":
    main()

## 2. Offline smoke test (~1 min) - confirms every module works before the long run

In [ ]:
!python tests/smoke_test.py

## 3. Download a dataset
Uses **140k Real and Fake Faces** (real = FFHQ photos, fake = StyleGAN) via `kagglehub`.
If it asks for Kaggle credentials: Kaggle -> Settings -> Create New Token, then set
`os.environ["KAGGLE_USERNAME"]` and `os.environ["KAGGLE_KEY"]` in a cell and re-run.

*Own data instead?* Skip this cell, set `DATA_SRC` to a folder containing `train/{real,fake}` and `valid/{real,fake}`.

In [ ]:
import kagglehub
DATA_SRC = kagglehub.dataset_download("xhlulu/140k-real-and-fake-faces")
print("Downloaded to:", DATA_SRC)

## 4. Build `data/train` and `data/val` (symlinks - no copying, no re-encoding)

In [ ]:
!python -m dfd.prepare_data --src "{DATA_SRC}" --out data --n_train 3000 --n_val 600

## 5. Train (about 10-20 min on a T4 for 6000 train images)

In [ ]:
!python -m dfd.train --train_dir data/train --val_dir data/val --epochs 10 --batch_size 32 --out models/detector_best.pt

## 6. Inspect results: metrics + Grad-CAM on validation images

In [ ]:
import json, glob, cv2, numpy as np, matplotlib.pyplot as plt
from dfd.pipeline import DeepfakePipeline

res = json.load(open("models/detector_best.json"))
best = max(res["history"], key=lambda h: h["auc"])
print({k: round(v, 3) for k, v in best.items()})

pipe = DeepfakePipeline("models/detector_best.pt")
files = [(f, "real") for f in sorted(glob.glob("data/val/real/*"))[:3]] + \
        [(f, "fake") for f in sorted(glob.glob("data/val/fake/*"))[:3]]
fig, axes = plt.subplots(2, len(files), figsize=(3 * len(files), 6))
for i, (f, truth) in enumerate(files):
    img = cv2.cvtColor(cv2.imread(f), cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (224, 224))
    from dfd.preprocessing import to_model_input
    from dfd.explain import overlay_heatmap
    r = pipe.explainer.explain(to_model_input(img))
    axes[0, i].imshow(img); axes[0, i].set_title(f"truth: {truth}"); axes[0, i].axis("off")
    axes[1, i].imshow(overlay_heatmap(img, r["heatmap"])); axes[1, i].axis("off")
    axes[1, i].set_title(f"P(fake)={r['probability_fake']:.2f}")
plt.tight_layout(); plt.show()

## 7. Launch the web demo (click the public `gradio.live` link)

In [ ]:
from app.gradio_app import build_demo
build_demo("models/detector_best.pt").launch(share=True, debug=True)

## 8. (Optional) Save the trained model to Google Drive
```python
from google.colab import drive; drive.mount('/content/drive')
!cp models/detector_best.pt models/detector_best.json /content/drive/MyDrive/
```
Or download `models/detector_best.pt` from the Files panel and put it in your local project's `models/` folder.